In [33]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
# Load the dataset using the pandas library function read_csv
data = pd.read_csv('e1_nutrients.csv')

In [ ]:
nutrient_cols = [c for c in data.columns if c != "Depth"]
plt.figure(figsize=(10, 6))
for col in nutrient_cols:
    plt.scatter(data["Depth"], data[col], s=35, alpha=0.75, label=col)

plt.xlabel("Depth")
plt.ylabel("Concentration")
plt.title("Scatter Plot of All Nutrient Data vs Depth")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Function to filter out outliers based on the IQR method for a specific column
def filter_by_column(data, column_name):
    # Check if the inputted column exists in the provided data
    if column_name in data.columns:
        # Check if the required depth column exists in the provided data
        if "Depth" not in data.columns:
            # If the depth column does not exist, raise a ValueError with an appropriate message
            raise ValueError("Column 'Depth' does not exist in the Data.")
        
        # Directly copy the data to a new set without modifying the original data
        data_new = data.copy()

        # Sort the data, using the depth column to ensure that the quantile calculations are performed correctly for each depth level
        data_new = data_new.sort_values("Depth", kind="mergesort").reset_index(drop=True)

        # Calculate the first and third quartiles (Q1 and Q3) for the specified column, grouped by depth
        q1 = data_new.groupby("Depth")[column_name].quantile(0.25)
        q3 = data_new.groupby("Depth")[column_name].quantile(0.75)

        # Calculate the interquartile range (IQR) for the specified column, grouped by depth
        iqr = q3 - q1

        # Using the IQR, calculating the lower and upper bounds to detect outliers for the specified column, grouped by depth
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Using the pandas map function, apply the calculated lower and upper bounds to the original data based on the depth values, creating two new series that represent the lower and upper bounds for each row in the original data
        lower_mapped = data_new["Depth"].map(lower_bound)
        upper_mapped = data_new["Depth"].map(upper_bound)

        # Filter the original data to include only rows where the values in the specified column are within the calculated lower and upper bounds, grouped by depth
        filtered_data = data_new[
            (data_new[column_name] >= lower_mapped) & 
            (data_new[column_name] <= upper_mapped)
        ]
        
        # Reset the indexes of the filtered data to keep the sequential nature of the data intact after the outliers are removed
        filtered_data = filtered_data.reset_index(drop=True)
        
        # Return the filtered data, which contains only the rows where the values in the specified column are within the calculated bounds for each depth level
        return filtered_data
    
    # If the specified column does not exist in the provided data, raise a ValueError
    else:
        # If the column does not exist, raise a ValueError
        raise ValueError(f"Column '{column_name}' does not exist in the Data.")

# List of columns to clean using the filter_by_column function
#columns_to_clean = ['NITRATE+NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE']
# Create a clean copy of the data to work on
#clean_data = data.copy()
# Definite iteration over the columns to clean, applying the filter function to each column.
#for col in columns_to_clean:
   #clean_data = filter_by_column(clean_data, col)


In [51]:
# Prepare the data for machine learning by separating the features (x) and the target variable (y), where the target variable is NITRITE


# Split the data into training and testing sets, using 80% of the data for training and 20% for testing
train_data, test_data = train_test_split(data, test_size=0.2, random_state=43)

columns_to_clean = ['NITRATE+NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE']
clean_train_data = train_data.copy()

for col in columns_to_clean:
    clean_train_data = filter_by_column(clean_train_data, col)

x_train = clean_train_data.drop('NITRITE', axis=1)
y_train = clean_train_data['NITRITE']

x_test = test_data.drop('NITRITE', axis=1)
y_test = test_data['NITRITE']

scalar = RobustScaler()

x_train_scaled = scalar.fit_transform(x_train)
x_test_scaled = scalar.transform(x_test)


lr_model = LinearRegression()
lr_model.fit(x_train_scaled, y_train)
lr_score_train = lr_model.score(x_train_scaled, y_train)
lr_score_test = lr_model.score(x_test_scaled, y_test)
print(f"Linear Regression R^2 Score (train): {lr_score_train}, R^2 Score (test): {lr_score_test}")

rf_model = RandomForestRegressor(random_state=43)
rf_model.fit(x_train_scaled, y_train)
rf_score_train = rf_model.score(x_train_scaled, y_train)
rf_score_test = rf_model.score(x_test_scaled, y_test)
print(f"Random Forest R^2 Score (train): {rf_score_train}, R^2 Score (test): {rf_score_test}")

nn_model = MLPRegressor(random_state=43, max_iter=1000)
nn_model.fit(x_train_scaled, y_train)
nn_score_train = nn_model.score(x_train_scaled, y_train)
nn_score_test = nn_model.score(x_test_scaled, y_test)
print(f"Neural Network R^2 Score (train): {nn_score_train}, R^2 Score (test): {nn_score_test}")

sample_input = x_test_scaled[[0]]
actual_value = y_test.iloc[0]

print(f"Actual NITRITE value: {actual_value}")
print(f"Linear Regression predicted: {lr_model.predict(sample_input)[0]}")
print(f"Random Forest predicted:     {rf_model.predict(sample_input)[0]}")
print(f"Neural Network predicted:    {nn_model.predict(sample_input)[0]}")

Linear Regression R^2 Score (train): 0.129222877034728, R^2 Score (test): 0.12889204264483067
Random Forest R^2 Score (train): 0.9288466955030527, R^2 Score (test): 0.47626860573673735
Neural Network R^2 Score (train): 0.4169580295605375, R^2 Score (test): 0.28102051295425035
Actual NITRITE value: 0.26
Linear Regression predicted: 0.028468115821514012
Random Forest predicted:     0.2831
Neural Network predicted:    0.26881872736513657


In [ ]:
# SHow both test and train mse
lr_error = mean_squared_error(y_test, lr_model.predict(x_test_scaled))
print(lr_error)

0.08137749332278993
[0.09492924 0.07467375 0.11830491 0.11179978 0.09356461]
